# Errors and silent failures

> Messages, their causes, and failures that give no message

::: {.callout-note appearance="simple" icon=false}
**Reference: look up a fact.** It describes what trufl does, without advising or teaching. Want to know why? Read an [explanation](../explanation/03_round.ipynb). [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

## Failures without a message

| Condition | Result | Guide |
|---|---|---|
| A value in the state is missing | It counts as the most urgent value of that criterion | [Use your own areas](../how-to/02_use_your_data.ipynb) |
| The survey has no data in an area, or no cell centre in it | `Prior` is a masked value, which `isna` does not detect until `.astype(float)` | [Use your own areas](../how-to/02_use_your_data.ipynb) |
| The survey covers part of an area | `Prior` describes that part only | [Use your own areas](../how-to/02_use_your_data.ipynb) |
| One measurement in an area is blank | `Max`, `Min` and `Standard Deviation` are missing for the area | [Add results](../how-to/07_add_results.ipynb) |
| A measurement has a `loc_id` that matches no area | It is ignored | [Add results](../how-to/07_add_results.ipynb) |
| Two areas have the same `loc_id` | Both are sampled under that identifier | [Use your own areas](../how-to/02_use_your_data.ipynb) |
| Directions or weights are in another order than the columns | They apply to the wrong criteria | [Set priorities](../how-to/03_set_priorities.ipynb) |
| Ranks or counts are in another order than the areas, including counts in a labelled `Series` | Samples go to the wrong areas | [Fit the budget](../how-to/04_fit_budget.ipynb) |
| The budget is small, the number of areas is not a multiple of 4, or `min` is above 0 | The total can differ from the budget, from no samples to several times the budget | [Fit the budget](../how-to/04_fit_budget.ipynb) |
| `rng` is an integer | Areas of the same shape get the same pattern of points | [Placement](01_methods.ipynb#placement-sampler.sample) |
| A point is west or north of the collector's raster | A cell on the other side is read | [Data formats](02_formats.ipynb#coordinate-systems) |
| A point is on a no-data cell of the collector's raster | The no-data code is returned as the value | [Data formats](02_formats.ipynb) |

## Messages

| Message contains | Raised when |
|---|---|
| `must not contain any negative numbers` | `LINEAR1`, `LINEAR3` or `VECTOR` meets a negative value |
| `must not be zero` | `LINEAR1` meets a benefit with a maximum of 0, or a cost with a minimum of 0 |
| `must not be equal to its minimum` | `LINEAR2` meets a constant criterion |
| `not normalized such that each element` | `n_method` is `None` and a value is outside 0 to 1 |
| `number of variables in the list` | The flags and the columns differ in number |
| `weight vector must be normalized` | The weights do not add up to 1 |
| `must sum to 1 in order to apply the EM` | `EM` follows a method other than `LINEAR3`, or the state has gaps |
| `not compatible with the PEARSON` | `VIC` is used with `PEARSON` |
| `negative ideal distance` | `TOPSIS` meets areas that are identical on every criterion |
| `'NoneType' object has no attribute 'upper'` | `s_method` is missing, or `w_method` when no weights are given |
| `could not convert string to float` | A criterion is text |
| `unsupported operand type(s) for /` | Ranks are a list where a NumPy array is needed |
| `Policy weighted not implemented` | A policy name has the wrong case |
| `Item wrong length` | There are fewer or more counts than areas |
| `Input shapes do not overlap raster` | An area is outside the survey, or in other coordinates |
| `out of bounds` | A point is east or south of the collector's raster |

## Not working

| Object | Status |
|---|---|
| `read_shapefile`, `read_geojson` | Not implemented. Return `None` |
| `anonymize_raster` | Raises `NameError` ([issue 10](https://github.com/franckalbinet/trufl/issues/10)) |
| `MoranICB` | Reports trends in random data |
| `PriorCB` in version 0.2.1 and earlier | Counts cells outside the area as 0. Priors are up to 12% low on the example map |

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

import tempfile
from pathlib import Path
import rasterio
from shapely.geometry import Point, box
from fastcore.test import test_fail
from trufl.utils import anonymize_raster
from trufl.reader import read_geojson

f = '../files/ground-truth-01-4326-simulated.tif'
areas = gridder(f)
table = lambda **cols: Optimizer(pd.DataFrame(cols))
at = lambda x, y: gpd.GeoDataFrame(geometry=[Point(x, y)], crs=areas.crs)
left, bottom, right, top = areas.total_bounds
middle = (bottom + top) / 2

# Messages, in the order of the table
for text, call in (
    ('must not contain any negative numbers', lambda: table(a=[-1., 2]).get_rank([True], [1], 'LINEAR1', s_method='CP')),
    ('must not be zero', lambda: table(a=[0., 2]).get_rank([False], [1], 'LINEAR1', s_method='CP')),
    ('must not be equal to its minimum', lambda: table(a=[2., 2]).get_rank([True], [1], 'LINEAR2', s_method='CP')),
    ('not normalized such that each element', lambda: table(a=[1., 2]).get_rank([True], [1], s_method='CP')),
    ('number of variables in the list', lambda: table(a=[1., 2], b=[1., 2]).get_rank([True], [1], 'LINEAR1', s_method='CP')),
    ('weight vector must be normalized', lambda: table(a=[1., 2]).get_rank([True], [2], 'LINEAR1', s_method='CP')),
    ('must sum to 1 in order to apply the EM', lambda: table(a=[1., 2, np.nan]).get_rank([True], None, 'LINEAR3', w_method='EM', s_method='CP')),
    ('not compatible with the PEARSON', lambda: table(a=[1., 2]).get_rank([True], None, 'LINEAR1', 'PEARSON', 'VIC', 'CP')),
    ('negative ideal distance', lambda: table(a=[1., 1]).get_rank([True], [1], s_method='TOPSIS')),
    ("'NoneType' object has no attribute 'upper'", lambda: table(a=[1., 2]).get_rank([True], [1], 'LINEAR1')),
    ('could not convert string to float', lambda: table(a=['farm', 'wood']).get_rank([True], [1], 'LINEAR2', s_method='CP')),
    ('unsupported operand type(s) for /', lambda: rank_to_sample([1, 2], 10)),
    ('Policy weighted not implemented', lambda: rank_to_sample(np.arange(1, 3), 10, policy='weighted')),
    ('Item wrong length', lambda: Sampler(areas).sample(np.ones(99, int))),
    ('Input shapes do not overlap raster', lambda: PriorCB(f)(0, State(at(left, middle), areas.to_crs(3857), []))),
    ('out of bounds', lambda: DataCollector(f).get_values(at(right + 0.001, middle))),
):
    try: call()
    except Exception as e: assert text in str(e), (text, str(e))
    else: raise AssertionError(f'no error: {text}')

# Failures without a message
test_eq(table(a=[np.nan, 1., 2.]).get_rank([False], [1], 'LINEAR2', s_method='CP').loc[0, 'rank'] <= 2, True)
with rasterio.open(f) as src: values, meta = src.read(1), src.meta.copy()
values[:20, :60] = -9999
meta.update(nodata=-9999)
holed = Path(tempfile.mkdtemp())/'holed.tif'
with rasterio.open(holed, 'w', **meta) as dst: dst.write(values, 1)
m = gpd.GeoDataFrame({'value': [1., np.nan, 3.]}, geometry=[Point(left, middle)] * 3, crs=areas.crs, index=pd.Index([0, 0, 999], name='loc_id'))
s = State(m, areas, [MaxCB(), CountCB(), PriorCB(holed)])()
test_eq(np.isnan(s.loc[0, 'Max']), True)
test_eq(s['Count'].sum(), 2)
test_eq(s['Prior'].isna().sum(), 0)
test_eq(s['Prior'].astype(float).isna().sum() > 0, True)
test_eq(len(Sampler(pd.concat([areas.head(1), areas.head(1)])).sample(np.array([1, 1]))), 2)
counts = pd.Series(0, index=areas.index); counts[0], counts[7] = 5, 2
wrong = Sampler(areas.iloc[::-1]).sample(counts).explode(index_parts=False)
test_eq(wrong.groupby(level=0).size().to_dict(), {0: 2, 7: 5})
test_eq(rank_to_sample(np.arange(1, 101), 20, 0, 'quantiles').sum(), 0)
test_eq(rank_to_sample(np.arange(1, 101), 20, 1, 'quantiles').sum(), 100)
p = Sampler(areas).sample(np.ones(len(areas), int), rng=0).get_coordinates().to_numpy() - areas.bounds.to_numpy()[:, :2]
test_eq(p.std(0).max() < 1e-9, True)
test_eq(len(DataCollector(f).get_values(at(left - 0.001, middle))), 1)
test_eq(DataCollector(holed).get_values(at(left + 0.001, top - 0.001))[0], -9999)
test_eq(read_geojson('any.geojson'), None)
test_fail(lambda: anonymize_raster(f, 0, 0), contains='src_fname')